In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet('/kaggle/input/datasets/zxcdfg/dataset-dt1/train.parquet')
df_valid = pd.read_parquet('/kaggle/input/datasets/zxcdfg/dataset-dt1/valid.parquet')

In [3]:
import sys
sys.path.append('/kaggle/input/datasets/zxcdfg/dataset-dt1')
from utils import weighted_pearson_correlation

In [4]:
df['dt1'] = df.groupby('seq_ix')['t1'].diff()

In [5]:
df["p1_minus_p11"] = df["p1"] - df["p11"]
df["p5_minus_p11"] = df["p5"] - df["p11"]
df["p4_minus_p11"] = df["p4"] - df["p11"]
df["p1_minus_p7"]  = df["p1"] - df["p7"]
df["p2_minus_p11"] = df["p2"] - df["p11"]
df['dp0_minus_dp2'] = df['dp0'] - df['dp2']

In [6]:
price_features = [f'p{i}' for i in range(12)]
volume_features = [f'v{i}' for i in range(12)]
p_features = [f'dp{i}' for i in range(4)]
v_features = [f'dv{i}' for i in range(4)]
features = price_features + volume_features + p_features + v_features + ['p1_minus_p11', 'p5_minus_p11', 'p4_minus_p11', 'p1_minus_p7', 'p2_minus_p11', 'dp0_minus_dp2'] 
df_nonan = df.dropna()

In [7]:
X_train = df_nonan[features]
y_train = df_nonan['dt1']

In [8]:
from sklearn.linear_model import LinearRegression
model = LinearRegression(fit_intercept=True)
model.fit(X_train, y_train)

LinearRegression()

In [9]:
df_valid["p1_minus_p11"] = df_valid["p1"] - df_valid["p11"]
df_valid["p5_minus_p11"] = df_valid["p5"] - df_valid["p11"]
df_valid["p4_minus_p11"] = df_valid["p4"] - df_valid["p11"]
df_valid["p1_minus_p7"]  = df_valid["p1"] - df_valid["p7"]
df_valid["p2_minus_p11"] = df_valid["p2"] - df_valid["p11"]
df_valid['dp0_minus_dp2'] = df_valid['dp0'] - df_valid['dp2']

In [10]:
df_valid['dt1_pred'] = model.predict(df_valid[features])

In [11]:
df_valid['t1_pred'] = np.zeros
df_valid['t1_pred'] = df_valid.groupby('seq_ix')['dt1_pred'].cumsum()

In [12]:
df_valid.loc[df['step_in_seq'] == 0, 'dt1_pred'] = df_valid['t1']

In [13]:
print(weighted_pearson_correlation(df_valid['t1_pred'], df_valid['t1']))

0.017585550445009757


In [14]:
df_valid.head(1001)

,seq_ix,step_in_seq,need_prediction,p0,p1,p2,p3,p4,p5,p6,...,t0,t1,p1_minus_p11,p5_minus_p11,p4_minus_p11,p1_minus_p7,p2_minus_p11,dp0_minus_dp2,dt1_pred,t1_pred
0,0,0,0,0.892038,-0.546700,0.293986,0.728240,0.115678,-0.018820,1.803203,...,-0.491662,-0.031387,0.803082,1.330963,1.465460,0.201186,1.643768,-0.366534,-0.031387,0.109178
1,0,1,0,1.059682,-0.658043,0.404717,0.754664,0.076604,-0.077862,2.056872,...,-0.375888,0.065718,0.772923,1.353104,1.507570,0.275415,1.835683,0.088385,0.130272,0.239450
2,0,2,0,1.035790,-0.832006,0.330873,0.705530,-0.006273,-0.172731,2.065579,...,-0.375888,0.071983,0.478288,1.137563,1.304022,0.101362,1.641167,-0.573888,0.019416,0.258867
3,0,3,0,1.307063,-0.853480,0.504922,0.773140,0.020074,-0.153660,2.279624,...,-0.375888,0.170653,0.624093,1.323913,1.497647,0.262016,1.982495,-0.233571,0.132831,0.391698
4,0,4,0,1.059682,-0.658043,0.404717,0.754664,0.076604,-0.077862,1.739312,...,-0.187944,0.270890,0.957371,1.537552,1.692017,-0.158405,2.020131,-0.266712,0.115301,0.506999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,0,996,1,0.892038,-0.546700,0.293986,0.728240,0.115678,-0.018820,1.132664,...,-0.187944,-0.579557,1.175989,1.703870,1.838367,0.526551,2.016675,0.013136,0.088465,-2.597509
997,0,997,1,0.895828,-0.406079,0.352151,0.768075,0.154930,0.069124,1.566414,...,0.115774,-0.421371,1.582891,2.058094,2.143900,0.943916,2.341122,-0.511706,0.181714,-2.415796
998,0,998,1,1.247505,-1.365257,0.315012,0.654930,-0.218133,-0.357496,1.713464,...,-0.260115,-0.634374,-0.365083,0.642679,0.782041,-0.795315,1.315186,-1.287890,-0.193201,-2.608997
999,0,999,1,1.014571,-1.018778,0.280911,0.665852,-0.096753,-0.249721,1.446104,...,-0.260115,-0.758104,0.072104,0.841161,0.994130,-0.538248,1.371793,0.088385,-0.093852,-2.702849


In [15]:
df_valid['dt1_true'] = df_valid.groupby('seq_ix')['t1'].diff()

In [16]:
df_valid = df_valid.dropna()

In [17]:
df_valid['t1_recon'] = np.zeros
df_valid['t1_recon'] = df_valid.groupby('seq_ix')['dt1_true'].cumsum()

In [18]:
df_valid['dt1_pred_2'] = df_valid['dt1_pred']
df_valid.loc[df['step_in_seq'] == 0, 'dt1_pred_2'] = df_valid['t1']
df_valid['t1_anchor'] = np.zeros
df_valid['t1_anchor'] = df_valid.groupby('seq_ix')['dt1_pred_2'].cumsum()

In [19]:
print(f'TRUE VS PREDICTED{weighted_pearson_correlation(df_valid['t1_pred'], df_valid['t1'])}, TRUE VS RECON{weighted_pearson_correlation(df_valid['t1_recon'], df_valid['t1'])}, RECON VS PREDICTED {weighted_pearson_correlation(df_valid['t1_pred'], df_valid['t1_recon'])}')

TRUE VS PREDICTED0.01758550260330677, TRUE VS RECON0.612971443950015, RECON VS PREDICTED 0.29889742527084134


In [20]:
print(weighted_pearson_correlation(df_valid['t1_pred'], df_valid['t1_anchor']))

0.7249861346247022


In [21]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score


In [22]:
STEP_TARGET = 100

P_COLS  = [f"p{i}" for i in range(12)]
V_COLS  = [f"v{i}" for i in range(12)]
DP_COLS = [f"dp{i}" for i in range(4)]
DV_COLS = [f"dv{i}" for i in range(4)]

FEATURE_COLS = P_COLS + V_COLS + DP_COLS + DV_COLS


In [23]:
def build_last_step_dataset(df):

    # Keep sequences that reach step 100
    valid_seqs = df.loc[df["step_in_seq"] == STEP_TARGET, "seq_ix"].unique()
    df = df[df["seq_ix"].isin(valid_seqs)]

    # Target
    y = (
        df[df["step_in_seq"] == STEP_TARGET]
        .set_index("seq_ix")["t1"]
    )

    # First 101 rows
    df_101 = df[df["step_in_seq"] <= STEP_TARGET]

    # Aggregate only our defined feature columns
    X = (
        df_101
        .groupby("seq_ix")[FEATURE_COLS]
        .agg(["mean", "std", "min", "max", "last"])
    )

    # Flatten column names
    X.columns = ["_".join(col) for col in X.columns]

    # Align
    X = X.loc[y.index]

    return X, y


In [24]:
X_train, y_train = build_last_step_dataset(df)

print("Train shape:", X_train.shape)


Train shape: (10721, 160)


In [25]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

feature_order = X_train.columns


In [26]:
X_valid, y_valid = build_last_step_dataset(df_valid)

# Force identical column order
X_valid = X_valid[feature_order]

print("Valid shape:", X_valid.shape)


Valid shape: (1444, 160)


In [27]:
pred_valid = model.predict(X_valid)

print("Validation R2:", r2_score(y_valid, pred_valid))
print("Validation Pearson:", np.corrcoef(y_valid, pred_valid)[0,1])
print("Pred std:", pred_valid.std())
print("True std:", y_valid.std())


Validation R2: -0.06789435453553505
Validation Pearson: -0.009372348077446116
Pred std: 0.44722527389935207
True std: 1.9958461969539465


In [28]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# -----------------------------
# Define feature columns
# -----------------------------
P_COLS  = [f"p{i}" for i in range(12)]
V_COLS  = [f"v{i}" for i in range(12)]
DP_COLS = [f"dp{i}" for i in range(4)]
DV_COLS = [f"dv{i}" for i in range(4)]

FEATURE_COLS = P_COLS + V_COLS + DP_COLS + DV_COLS

STEP_TARGET = 100

# -----------------------------
# TRAIN DATA
# -----------------------------

# Step 0 features
df_step0_train = df[df["step_in_seq"] == 0].set_index("seq_ix")

# Step 100 target
df_step100_train = df[df["step_in_seq"] == STEP_TARGET].set_index("seq_ix")

# Keep only sequences that have both step 0 and step 100
common_train = df_step0_train.index.intersection(df_step100_train.index)

X_train = df_step0_train.loc[common_train, FEATURE_COLS]
y_train = df_step100_train.loc[common_train, "t1"]

print("Train shape:", X_train.shape)

# -----------------------------
# VALIDATION DATA
# -----------------------------

df_step0_valid = df_valid[df_valid["step_in_seq"] == 0].set_index("seq_ix")
df_step100_valid = df_valid[df_valid["step_in_seq"] == STEP_TARGET].set_index("seq_ix")

common_valid = df_step0_valid.index.intersection(df_step100_valid.index)

X_valid = df_step0_valid.loc[common_valid, FEATURE_COLS]
y_valid = df_step100_valid.loc[common_valid, "t1"]

print("Valid shape:", X_valid.shape)

# -----------------------------
# MODEL
# -----------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

pred_valid = model.predict(X_valid)

# -----------------------------
# EVALUATION
# -----------------------------

print("Validation R2:", r2_score(y_valid, pred_valid))
print("Validation Pearson:", np.corrcoef(y_valid, pred_valid)[0,1])
print("Pred std:", pred_valid.std())
print("True std:", y_valid.std())


Train shape: (10721, 32)
Valid shape: (0, 32)


ValueError: Found array with 0 sample(s) (shape=(0, 32)) while a minimum of 1 is required by StandardScaler.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Use only step 100 rows
df_last_train = df[df["step_in_seq"] == 100]
df_last_valid = df_valid[df_valid["step_in_seq"] == 100]

X_train = df_last_train[FEATURE_COLS]
y_train = df_last_train["t1"]

X_valid = df_last_valid[FEATURE_COLS]
y_valid = df_last_valid["t1"]

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

pred = rf.predict(X_valid)

print("R2:", r2_score(y_valid, pred))
print("Pearson:", np.corrcoef(y_valid, pred)[0,1])
print("Pred std:", pred.std())
print("True std:", y_valid.std())


In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

# ==========================================================
# 1️⃣ Define feature columns
# ==========================================================

P_COLS  = [f"p{i}" for i in range(12)]
V_COLS  = [f"v{i}" for i in range(12)]
DP_COLS = [f"dp{i}" for i in range(4)]
DV_COLS = [f"dv{i}" for i in range(4)]

FEATURE_COLS = P_COLS + V_COLS + DP_COLS + DV_COLS
STEP_TARGET = 100


# ==========================================================
# 2️⃣ Structured feature builder
# ==========================================================

def build_structured_features(df, step_target=100):

    valid_seqs = df.loc[df["step_in_seq"] == step_target, "seq_ix"].unique()
    df = df[df["seq_ix"].isin(valid_seqs)]

    grouped = df.groupby("seq_ix")

    rows = []

    for seq, g in grouped:

        g = g[g["step_in_seq"] <= step_target].sort_values("step_in_seq")

        row = {}

        for col in FEATURE_COLS:

            series = g[col].values

            # last value
            row[f"{col}_last"] = series[-1]

            # last 5 mean
            row[f"{col}_mean5"] = series[-5:].mean()

            # last 20 mean
            row[f"{col}_mean20"] = series[-20:].mean()

            # cumulative sum
            row[f"{col}_cumsum"] = series.sum()

            # slope over full window
            t = np.arange(len(series))
            row[f"{col}_slope"] = np.polyfit(t, series, 1)[0]

        row["seq_ix"] = seq
        rows.append(row)

    X = pd.DataFrame(rows).set_index("seq_ix")

    y = (
        df[df["step_in_seq"] == step_target]
        .set_index("seq_ix")["t1"]
    )

    X = X.loc[y.index]

    return X, y


# ==========================================================
# 3️⃣ Build train dataset
# ==========================================================

X_train, y_train = build_structured_features(df, STEP_TARGET)

print("Train shape:", X_train.shape)


# ==========================================================
# 4️⃣ Train LightGBM
# ==========================================================

model = LGBMRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


# ==========================================================
# 5️⃣ Build validation dataset
# ==========================================================

X_valid, y_valid = build_structured_features(df_valid, STEP_TARGET)

# Ensure identical column order
X_valid = X_valid[X_train.columns]

print("Valid shape:", X_valid.shape)


# ==========================================================
# 6️⃣ Evaluate
# ==========================================================

pred_valid = model.predict(X_valid)

print("Validation R2:", r2_score(y_valid, pred_valid))
print("Validation Pearson:", np.corrcoef(y_valid, pred_valid)[0,1])
print("Pred std:", pred_valid.std())
print("True std:", y_valid.std())


# ==========================================================
# 7️⃣ Feature Importance
# ==========================================================

importance = pd.Series(model.feature_importances_, index=X_train.columns)
print("\nTop 20 Important Features:")
print(importance.sort_values(ascending=False).head(20))


In [ ]:
# Build validation structured features
X_valid, y_valid = build_structured_features(df_valid, step_target=100)

# Make sure columns match training
X_valid = X_valid[X_train.columns]

pred_valid = model.predict(X_valid)


In [ ]:
pred_valid = model.predict(X_valid)


In [ ]:
from sklearn.metrics import r2_score

print("Validation R2:", r2_score(y_valid, pred_valid))


In [ ]:
import numpy as np

pearson = np.corrcoef(y_valid, pred_valid)[0, 1]
print("Validation Pearson:", pearson)


In [ ]:
print(X_valid.shape)
print(y_valid.shape)


In [ ]:
from sklearn.metrics import r2_score
import numpy as np

pred_valid = model.predict(X_valid)

print("Validation R2:", r2_score(y_valid, pred_valid))
print("Validation Pearson:", np.corrcoef(y_valid, pred_valid)[0,1])
print("Pred std:", pred_valid.std())
print("True std:", y_valid.std())


In [ ]:
df_valid['t1'].head(101)